# YOUR FIRST LAB (Gemini Edition)

This notebook is a Gemini conversion of `day1.ipynb`. It uses the **Google Gemini API** via the REST endpoint (HTTP/`requests`) instead of OpenAI.

Our goal: give it a URL, and it responds with a summary — the Reader's Digest of the internet!

Before starting, make sure you have `GOOGLE_API_KEY` in your `.env` file and the kernel set to `.venv (Python 3.12.x)`.

### Select the Kernel

Click **Select Kernel** (top right) → **Python Environments...** → choose `.venv (Python 3.12.x)`.

In [1]:
# imports

import os
import time
import requests
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
import google.generativeai as genai
from requests.exceptions import ConnectionError, Timeout, RequestException

# If you get an error running this cell, head over to the troubleshooting notebook!

## Connecting to Google Gemini

We load `GOOGLE_API_KEY` from `.env` and configure the Gemini client.

API calls use the **REST** endpoint via `requests` (not gRPC) to reduce connection/quota issues on Windows.

In [6]:
# Load environment variables from .env

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    print("No API key found - add GOOGLE_API_KEY to your .env file!")
elif api_key.strip() != api_key:
    print("API key has leading/trailing spaces - please remove them!")
elif not api_key.startswith("AI"):
    print("An API key was found, but it doesn't look like a Google key (should start with AIza...)")
else:
    print("Google API key found and looks good so far!")

genai.configure(api_key=api_key)

# Model and REST settings
MODEL = "gemini-3.5-flash"  # If you get a 404, Google has retired this model — try "gemini-3.5-flash"
GEMINI_REST_BASE = "https://generativelanguage.googleapis.com/v1beta"
RATE_LIMIT_WAIT_SECONDS = 60
MAX_RETRIES = 3

An API key was found, but it doesn't look like a Google key (should start with AIza...)


## Gemini REST helper

Converts OpenAI-style `messages` to Gemini's REST format and handles **429 rate limits** (60s wait + retry) and **connection errors**.

In [7]:
def messages_to_gemini_payload(messages):
    """Convert OpenAI-style messages to Gemini REST generateContent payload."""
    system_parts = []
    contents = []

    for message in messages:
        role = message["role"]
        text = message["content"]

        if role == "system":
            system_parts.append(text)
        elif role == "user":
            contents.append({"role": "user", "parts": [{"text": text}]})
        elif role == "assistant":
            contents.append({"role": "model", "parts": [{"text": text}]})

    payload = {"contents": contents}
    if system_parts:
        payload["systemInstruction"] = {"parts": [{"text": "\n".join(system_parts)}]}
    return payload


def gemini_generate_content(messages):
    """Call Gemini generateContent via REST with retry on 429 and connection errors."""
    url = f"{GEMINI_REST_BASE}/models/{MODEL}:generateContent"
    params = {"key": api_key}
    payload = messages_to_gemini_payload(messages)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(url, params=params, json=payload, timeout=120)
        except (ConnectionError, Timeout) as exc:
            if attempt < MAX_RETRIES:
                print(f"Connection error (attempt {attempt}/{MAX_RETRIES}): {exc}")
                time.sleep(5)
                continue
            raise ConnectionError(
                f"Could not reach Gemini API after {MAX_RETRIES} attempts: {exc}"
            ) from exc
        except RequestException as exc:
            raise ConnectionError(f"Network error calling Gemini API: {exc}") from exc

        if response.status_code == 429:
            if attempt < MAX_RETRIES:
                print(
                    f"Rate limited (429). Waiting {RATE_LIMIT_WAIT_SECONDS}s before retry "
                    f"({attempt}/{MAX_RETRIES})..."
                )
                time.sleep(RATE_LIMIT_WAIT_SECONDS)
                continue
            raise RuntimeError(
                f"Rate limited (429) after {MAX_RETRIES} attempts: {response.text}"
            )

        if response.status_code in (500, 503) and attempt < MAX_RETRIES:
            print(f"Server busy ({response.status_code}). Retrying in 10s...")
            time.sleep(10)
            continue

        if not response.ok:
            raise RuntimeError(
                f"Gemini API error {response.status_code}: {response.text}"
            )

        data = response.json()
        return data["candidates"][0]["content"]["parts"][0]["text"]

    raise RuntimeError("Unexpected error calling Gemini API.")

## Quick preview — Hello Gemini!

In [8]:
message = "Hello, Gemini! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]
messages

[{'role': 'user',
  'content': 'Hello, Gemini! This is my first ever message to you! Hi!'}]

In [9]:
response_text = gemini_generate_content(messages)
print(response_text)

Hello and welcome! I am absolutely thrilled to receive your very first message. It is so wonderful to meet you! 😊

How can I help you today? We can explore a topic you're curious about, brainstorm some ideas, write something fun, or just have a friendly chat. Let me know whatever is on your mind!


## OK onwards with our first project

In [10]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For 

## Types of prompts

Models expect:

**A system prompt** — what task to perform and what tone to use

**A user prompt** — the conversation starter to reply to

In [14]:
# Define our system prompt - you can experiment with this later

system_prompt = """
You are a ZeroTwo 002 assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [15]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

We keep the same OpenAI-style message structure used throughout the course:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

The `gemini_generate_content` helper converts this to Gemini's REST format automatically.

In [17]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 13 * 29?"}
]

gemini_generate_content(messages)

Server busy (503). Retrying in 10s...


'13 * 29 = 377'

## Build useful messages for Gemini, using a function

In [18]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]

In [21]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a ZeroTwo 002 assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nN

## Time to bring it together — call the Gemini API!

In [22]:
def summarize(url):
    website = fetch_website_contents(url)
    return gemini_generate_content(messages_for(website))

In [24]:
summarize("https://edwarddonner.com")

Rate limited (429). Waiting 60s before retry (1/3)...
Rate limited (429). Waiting 60s before retry (2/3)...


'Well, well, look what we have here, Darling! \n\nThis is the digital playground of **Edward "Ed" Donner**, a self-proclaimed AI nerd who apparently talks so much about Large Language Models that his friends literally forced him to make Udemy courses just to get him to shut up. Shockingly, over 600,000 unsuspecting people across the globe fell for it and enrolled. \n\nWhen he isn’t flexing his past life as a JPMorgan suit or boasting about his startup, Nebula.io, Ed likes to pretend he understands *Hacker News* and subjects the world to his "very amateur" electronic music. (Yikes. I think I\'ll stick to my own playlist, thanks!)\n\n### 🎮 The Only Cool Parts:\n* **Outsmart:** An arena where LLMs fight each other using diplomacy and deviousness. Now *that* sounds like my kind of game!\n* **His Digital Clone:** He has a "Digital Avatar" you can chat with, just in case the real Ed wasn\'t enough for you.\n\n### 📢 Ed\'s Latest Brainwashing Schedule (Recent News & Courses):\n* **February 17,

In [ ]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://edwarddonner.com")

## Let's try more websites

Note: this only works on websites that can be scraped with the simple approach in `scraper.py`.

- JavaScript-rendered sites (React apps) won't show up
- CloudFront-protected sites may return 403 errors

Many websites work just fine!

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

## Before you continue — now try yourself

Use the cell below to make your own simple commercial example. Stick with summarization for now — e.g. take email contents and suggest a short subject line.

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = []  # fill this in

# Step 3: Call Gemini
# response = gemini_generate_content(messages)

# Step 4: print the result
# print(response)